<a href="https://colab.research.google.com/github/Korchomnyi-Ivan/Korchomnyi-Ivan-Machine-3-15/blob/main/LR_10_Korchomnyi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install scikit-surprise

1. Доцільна реалізація

In [ ]:
from surprise import Dataset, Reader
from surprise import SVD, KNNBasic
from surprise.model_selection import cross_validate, GridSearchCV
from surprise import accuracy
from surprise.model_selection import train_test_split

import sys
import io

In [ ]:
!wget --no-check-certificate https://files.grouplens.org/datasets/movielens/ml-100k.zip

!unzip -o ml-100k.zip

--2026-04-21 19:47:26--  https://files.grouplens.org/datasets/movielens/ml-100k.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4924029 (4.7M) [application/zip]
Saving to: ‘ml-100k.zip’

ml-100k.zip         100%[===================>]   4.70M  11.4MB/s    in 0.4s    

2026-04-21 19:47:26 (11.4 MB/s) - ‘ml-100k.zip’ saved [4924029/4924029]

Archive:  ml-100k.zip
   creating: ml-100k/
  inflating: ml-100k/allbut.pl       
  inflating: ml-100k/mku.sh          
  inflating: ml-100k/README          
  inflating: ml-100k/u.data          
  inflating: ml-100k/u.genre         
  inflating: ml-100k/u.info          
  inflating: ml-100k/u.item          
  inflating: ml-100k/u.occupation    
  inflating: ml-100k/u.user          
  inflating: ml-100k/u1.base         
  inflating: ml-100k/u1.test         
  inflating: ml-100k/u2.ba

In [ ]:
import pandas as pd

data = pd.read_csv('ml-100k/u.data', sep='\t', names=['user_id', 'movie_id', 'rating', 'timestamp'])
data.head()

,user_id,movie_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [ ]:
reader = Reader(rating_scale=(1, 5))
data_surprise = Dataset.load_from_df(data[['user_id', 'movie_id', 'rating']], reader)

In [ ]:
trainset, testset = train_test_split(data_surprise, test_size=0.2)

# 3. GridSearchCV для SVD
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

grid_search_svd = GridSearchCV(SVD, param_grid_svd, measures=['rmse', 'mae'], cv=3)
grid_search_svd.fit(data_surprise)

print("Best RMSE for SVD:", grid_search_svd.best_score['rmse'])
print("Best params for SVD:", grid_search_svd.best_params['rmse'])

# 4. GridSearchCV для KNNBasic
param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [True, False]
    }
}

# Приглушення виводу similarity matrix
sys_stdout_backup = sys.stdout
sys.stdout = io.StringIO()

grid_search_knn = GridSearchCV(KNNBasic, param_grid_knn, measures=['rmse', 'mae'], cv=3)
grid_search_knn.fit(data_surprise)

# Повернення нормального виводу
sys.stdout = sys_stdout_backup

print("Best RMSE for KNNBasic:", grid_search_knn.best_score['rmse'])
print("Best params for KNNBasic:", grid_search_knn.best_params['rmse'])

# 5. Найкращий SVD
best_svd = SVD(**grid_search_svd.best_params['rmse'])
best_svd.fit(trainset)
predictions_svd = best_svd.test(testset)
print("Test RMSE (SVD):", accuracy.rmse(predictions_svd))

# 6. Найкращий KNNBasic
best_knn = KNNBasic(**grid_search_knn.best_params['rmse'])
best_knn.fit(trainset)
predictions_knn = best_knn.test(testset)
print("Test RMSE (KNNBasic):", accuracy.rmse(predictions_knn))

Best RMSE for SVD: 0.9297385607083234
Best params for SVD: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}
Best RMSE for KNNBasic: 1.019756286476724
Best params for KNNBasic: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'pearson', 'user_based': True}}
RMSE: 0.9275
Test RMSE (SVD): 0.9274716354821173
Computing the pearson similarity matrix...
Done computing similarity matrix.
RMSE: 1.0219
Test RMSE (KNNBasic): 1.0219031101208453


In [ ]:
best_svd = grid_search_svd.best_estimator['mae']
best_knn = grid_search_knn.best_estimator['mae']

print(f"Найкращий MAE для SVD: {grid_search_svd.best_score['mae']}")
print(f"Найкращий MAE для KNNBasic: {grid_search_knn.best_score['mae']}")

# Вибір моделі з найменшим MAE
if grid_search_svd.best_score['mae'] < grid_search_knn.best_score['mae']:
    best_model = best_svd
    print("Вибраний алгоритм: SVD")
else:
    best_model = best_knn
    print("Вибраний алгоритм: KNNBasic")



Найкращий MAE для SVD: 0.7323180570357515
Найкращий MAE для KNNBasic: 0.808598813291105
Вибраний алгоритм: SVD


In [ ]:
for i, (uid, iid, rating) in enumerate(trainset.all_ratings()):
    print(uid, iid, rating)
    if i > 10:
        break

0 0 5.0
0 582 5.0
0 951 4.0
0 450 5.0
0 1175 5.0
0 318 3.0
0 908 1.0
0 807 4.0
0 838 1.0
0 285 5.0
0 519 3.0
0 603 5.0


In [ ]:
best_model.fit(trainset)

# Беремо реального користувача з початкового DataFrame
user_id = data['user_id'].iloc[0]

# Якщо trainset зберігає id як рядки, розкоментуй:
# user_id = str(data['user_id'].iloc[0])

# Перевірка, чи є користувач у trainset
inner_uid = trainset.to_inner_uid(user_id)
user_ratings = trainset.ur[inner_uid]

print(f"Кількість рецензій користувача {user_id}: {len(user_ratings)}")

# Усі фільми та вже оцінені фільми
all_items = set(trainset.all_items())
rated_items = set(item for item, _ in user_ratings)
unrated_items = all_items - rated_items

# Прогноз рейтингів для неоцінених фільмів
predictions = [
    (trainset.to_raw_iid(item), best_model.predict(user_id, trainset.to_raw_iid(item)).est)
    for item in unrated_items
]

# Сортування за спаданням прогнозованого рейтингу
predictions.sort(key=lambda x: x[1], reverse=True)

# Виведення топ-10 рекомендацій
print("Топ-10 фільмів, рекомендованих для користувача:")
for item_id, rating in predictions[:10]:
    print(f"Фільм {item_id} з прогнозованим рейтингом {rating:.2f}")

Кількість рецензій користувача 196: 32
Топ-10 фільмів, рекомендованих для користувача:
Фільм 1449 з прогнозованим рейтингом 4.56
Фільм 923 з прогнозованим рейтингом 4.47
Фільм 251 з прогнозованим рейтингом 4.47
Фільм 603 з прогнозованим рейтингом 4.46
Фільм 178 з прогнозованим рейтингом 4.43
Фільм 272 з прогнозованим рейтингом 4.43
Фільм 64 з прогнозованим рейтингом 4.38
Фільм 488 з прогнозованим рейтингом 4.38
Фільм 661 з прогнозованим рейтингом 4.36
Фільм 1021 з прогнозованим рейтингом 4.35
